# Phase 2 — Master Table Builder

**Owner:** Member 1 (Data & EDA Lead)  
**Goal:** Build the canonical `master_creatives.csv` (one row per `creative_id`) that Members 2/3/4 will use for everything.  
**Time budget:** 60–90 min.  
**Inputs:** All 6 CSVs in `data/`, schema from Phase 1.  
**Outputs:** `master_creatives.csv` + `MASTER_SCHEMA.md`

## What's in the master table

One row per creative, with:
- IDs and dimensional context (advertiser, campaign, vertical, etc.)
- All visual/content features from `creatives.csv`
- All lifetime + fatigue metrics from `creative_summary.csv`
- Recomputed metrics (sanity-checked against the pre-computed ones)
- Derived flags useful for the demo (`is_fatigued_recoverable`, `health_score`, etc.)

## Strategic note

Phase 1 revealed that `creative_status='fatigued'` actually means **"former top performer that declined"**, not "bad performer." We encode this insight directly into the master table via the `is_fatigued_recoverable` flag — fatigued creatives whose lifetime metrics still beat stable ones. This is the Tier-3 pitch hook: *"Don't pause fatigued creatives — refresh them."*

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 240)

DATA_DIR = Path("")
OUT_DIR  = Path(".")  # write outputs alongside the notebook

ID_COLS   = {"advertiser_id", "campaign_id", "creative_id"}
DATE_COLS = {"date", "start_date", "end_date", "creative_launch_date"}

def smart_load(path):
    df = pd.read_csv(path, low_memory=False)
    for c in df.columns:
        if c in ID_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
        elif c in DATE_COLS:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

advertisers = smart_load(DATA_DIR / "advertisers.csv")
campaigns   = smart_load(DATA_DIR / "campaigns.csv")
creatives   = smart_load(DATA_DIR / "creatives.csv")
summary     = smart_load(DATA_DIR / "creative_summary.csv")

for name, df in [("advertisers", advertisers), ("campaigns", campaigns),
                 ("creatives", creatives), ("creative_summary", summary)]:
    print(f"  {name:<20} {len(df):>6,} rows \u00d7 {df.shape[1]:>3} cols")

  advertisers              36 rows ×   4 cols
  campaigns               180 rows ×  14 cols
  creatives             1,080 rows ×  32 cols
  creative_summary      1,080 rows ×  59 cols


## 1. Resolve duplicate columns BEFORE joining

`creatives` and `creative_summary` both have `vertical`, `format`, `cta_text`, `dominant_color`, etc. After joining we'd get `vertical_x` / `vertical_y` which is ugly and error-prone. We pick the authoritative source for each shared column and drop the duplicate from the other table.

In [2]:
# What columns overlap between creatives and creative_summary?
shared = set(creatives.columns) & set(summary.columns) - {"creative_id", "campaign_id"}
print(f"Columns in both `creatives` and `creative_summary` ({len(shared)}):\n  {sorted(shared)}\n")

# Sanity check: do the values actually match? If they do, we can drop from creatives safely.
merged_check = creatives[["creative_id"] + sorted(shared)].merge(
    summary[["creative_id"] + sorted(shared)],
    on="creative_id", suffixes=("_cr", "_su"))

for col in sorted(shared):
    a = merged_check[f"{col}_cr"]
    b = merged_check[f"{col}_su"]
    # Compare allowing NaN==NaN
    mismatch = (a != b) & ~(a.isna() & b.isna())
    n_mismatch = mismatch.sum()
    flag = " \u26a0 MISMATCH" if n_mismatch > 0 else ""
    print(f"  {col:<24} mismatches: {n_mismatch}{flag}")

# If everything matches: keep summary's version (it's our spine), drop from creatives.
creatives_lite = creatives.drop(columns=list(shared))
print(f"\nDropped {len(shared)} duplicate columns from creatives. Keeping versions from creative_summary.")
print(f"creatives_lite: {creatives_lite.shape}")

Columns in both `creatives` and `creative_summary` (30):
  ['advertiser_name', 'app_name', 'asset_file', 'brand_visibility_score', 'clutter_score', 'copy_length_chars', 'creative_launch_date', 'cta_text', 'dominant_color', 'duration_sec', 'emotional_tone', 'faces_count', 'format', 'has_discount_badge', 'has_gameplay', 'has_price', 'has_ugc_style', 'headline', 'height', 'hook_type', 'language', 'motion_score', 'novelty_score', 'product_count', 'readability_score', 'subhead', 'text_density', 'theme', 'vertical', 'width']

  advertiser_name          mismatches: 0
  app_name                 mismatches: 0
  asset_file               mismatches: 0
  brand_visibility_score   mismatches: 0
  clutter_score            mismatches: 0
  copy_length_chars        mismatches: 0
  creative_launch_date     mismatches: 0
  cta_text                 mismatches: 0
  dominant_color           mismatches: 0
  duration_sec             mismatches: 0
  emotional_tone           mismatches: 0
  faces_count          

## 2. Build the master table (left joins from `creative_summary`)

The spine is `creative_summary` because it's already 1-row-per-creative and has the rich metrics. We left-join everything else.

**Invariant we assert at every step:** `len(master) == len(summary)`. If a join causes the row count to change, we've found a bug.

In [3]:
EXPECTED_ROWS = len(summary)
print(f"Target row count: {EXPECTED_ROWS:,}\n")

# Step 1: spine = summary
master = summary.copy()
print(f"  After spine:                     {len(master):>5,} rows")
assert len(master) == EXPECTED_ROWS

# Step 2: add the asset_file and any creatives-only columns
master = master.merge(creatives_lite, on=["creative_id", "campaign_id"], how="left")
print(f"  After + creatives:               {len(master):>5,} rows")
assert len(master) == EXPECTED_ROWS, "creatives join exploded — check for duplicate creative_ids"

# Step 3: add campaign-level context
# campaigns has its own `vertical` and `advertiser_name` — disambiguate with prefix
campaign_cols = ["campaign_id", "advertiser_id", "objective", "primary_theme",
                 "target_age_segment", "target_os", "countries",
                 "start_date", "end_date", "daily_budget_usd", "kpi_goal"]
campaign_cols = [c for c in campaign_cols if c in campaigns.columns]
master = master.merge(
    campaigns[campaign_cols].rename(columns={
        "start_date": "campaign_start_date",
        "end_date":   "campaign_end_date",
    }),
    on="campaign_id", how="left"
)
print(f"  After + campaigns:               {len(master):>5,} rows")
assert len(master) == EXPECTED_ROWS, "campaigns join exploded"

# Step 4: add advertiser context
master = master.merge(
    advertisers[["advertiser_id", "hq_region"]],
    on="advertiser_id", how="left"
)
print(f"  After + advertisers:             {len(master):>5,} rows")
assert len(master) == EXPECTED_ROWS, "advertisers join exploded"

print(f"\n\u2713 Master table built: {len(master):,} rows \u00d7 {master.shape[1]} cols")

Target row count: 1,080

  After spine:                     1,080 rows
  After + creatives:               1,080 rows
  After + campaigns:               1,080 rows
  After + advertisers:             1,080 rows

✓ Master table built: 1,080 rows × 70 cols


## 3. Recompute key metrics (don't blindly trust pre-computed ones)

We re-derive `ctr`, `cvr`, `cpc`, `cpa`, `roas` from raw totals. Then we compare against the pre-computed `overall_ctr` etc. Mismatches reveal data quality issues we need to know about.

In [4]:
def safe_div(num, den):
    """Divide; 0 or NaN denominator -> NaN (never inf)."""
    den_clean = den.replace(0, np.nan)
    return (num / den_clean)

master["ctr_recomputed"]  = safe_div(master["total_clicks"],      master["total_impressions"])
master["cvr_recomputed"]  = safe_div(master["total_conversions"], master["total_clicks"])
master["cpc_usd"]         = safe_div(master["total_spend_usd"],   master["total_clicks"])
master["cpa_usd"]         = safe_div(master["total_spend_usd"],   master["total_conversions"])
master["roas_recomputed"] = safe_div(master["total_revenue_usd"], master["total_spend_usd"])
master["ipm"]             = safe_div(master["total_conversions"], master["total_impressions"]) * 1000

# Sanity check vs pre-computed
checks = [("ctr",  "overall_ctr",  "ctr_recomputed"),
          ("cvr",  "overall_cvr",  "cvr_recomputed"),
          ("roas", "overall_roas", "roas_recomputed"),
          ("ipm",  "overall_ipm",  "ipm")]

print("Recomputed vs pre-computed metric agreement:\n")
for label, precomp, recomp in checks:
    diff = (master[precomp] - master[recomp]).abs()
    n_diff = (diff > 1e-4).sum()
    max_diff = diff.max()
    status = "\u2713" if n_diff == 0 else "\u26a0"
    print(f"  {status}  {label:<5}  mismatches: {n_diff:>4}   max abs diff: {max_diff:.6f}")

Recomputed vs pre-computed metric agreement:

  ✓  ctr    mismatches:    0   max abs diff: 0.000000
  ✓  cvr    mismatches:    0   max abs diff: 0.000000
  ✓  roas   mismatches:    0   max abs diff: 0.000000
  ✓  ipm    mismatches:    0   max abs diff: 0.000000


## 4. Add derived flags & scores (the Tier-3 demo hooks)

These are the columns Members 3/4 will use to power the "interesting" parts of the demo: which creatives to scale, pause, refresh, or twin-match.

In [5]:
# --- Vertical-relative percentile rank for ROAS (so we don't unfairly punish low-ROAS verticals) ---
master["roas_pct_rank_in_vertical"] = (
    master.groupby("vertical")["overall_roas"]
          .rank(pct=True, na_option="keep")
          .round(3)
)

master["ctr_pct_rank_in_vertical"] = (
    master.groupby("vertical")["overall_ctr"]
          .rank(pct=True, na_option="keep")
          .round(3)
)

# --- Health Score (transparent, replaces the synthetic perf_score) ---
# Blend: 40% lifetime-CTR rank, 40% lifetime-ROAS rank, 20% (1 + ctr_decay_pct) capped at [0, 1]
# Rationale: rewards creatives that perform well AND aren't decaying.
decay_component = (1 + master["ctr_decay_pct"]).clip(0, 1)  # decay is negative; -0.5 -> 0.5
master["health_score"] = (
    0.40 * master["ctr_pct_rank_in_vertical"].fillna(0.5) +
    0.40 * master["roas_pct_rank_in_vertical"].fillna(0.5) +
    0.20 * decay_component.fillna(0.5)
).round(3)

# --- The KEY flag: fatigued but recoverable ---
# Phase-1 finding: creative_status='fatigued' means "former top performer that declined".
# A fatigued creative is RECOVERABLE if its lifetime metrics still beat the stable median.
stable_ctr_median  = master.loc[master["creative_status"] == "stable", "overall_ctr"].median()
stable_roas_median = master.loc[master["creative_status"] == "stable", "overall_roas"].median()

master["is_fatigued_recoverable"] = (
    (master["creative_status"] == "fatigued") &
    (master["overall_ctr"] > stable_ctr_median) &
    (master["overall_roas"] > stable_roas_median)
).astype(int)

# --- Action recommendation (rules-based v0; Member 2 will replace with a model later) ---
def recommend_action(row):
    if row["creative_status"] == "top_performer" and row["health_score"] >= 0.7:
        return "SCALE"
    if row["is_fatigued_recoverable"] == 1:
        return "REFRESH"  # was a winner, decayed; copy its DNA into a new creative
    if row["creative_status"] == "fatigued":
        return "PAUSE"
    if row["creative_status"] == "underperformer":
        return "PAUSE"
    if row["creative_status"] == "stable" and row["health_score"] >= 0.6:
        return "HOLD"
    return "TEST"  # default: keep monitoring

master["recommended_action"] = master.apply(recommend_action, axis=1)

print("Health score distribution by creative_status:")
print(master.groupby("creative_status")["health_score"].describe()[["mean", "50%", "min", "max"]].round(3))

print("\nRecommended action distribution:")
print(master["recommended_action"].value_counts())

print(f"\nFatigued-but-recoverable creatives: {master['is_fatigued_recoverable'].sum()} "
      f"of {(master['creative_status'] == 'fatigued').sum()} fatigued total")

Health score distribution by creative_status:
                  mean    50%    min    max
creative_status                            
fatigued         0.553  0.558  0.243  0.820
stable           0.433  0.421  0.050  0.866
top_performer    0.770  0.783  0.560  0.875
underperformer   0.171  0.154  0.068  0.367

Recommended action distribution:
recommended_action
TEST       588
PAUSE      209
HOLD       161
REFRESH     85
SCALE       37
Name: count, dtype: int64

Fatigued-but-recoverable creatives: 85 of 199 fatigued total


## 5. Order columns for downstream readability

Convention from `eda_insights.md`: IDs first, dimensions second, raw metrics third, derived metrics last.

In [6]:
COL_ORDER = [
    # === IDs ===
    "creative_id", "campaign_id", "advertiser_id",
    
    # === Naming/labels ===
    "advertiser_name", "app_name",
    
    # === Dimensions ===
    "vertical", "format", "language", "theme", "hook_type",
    "cta_text", "dominant_color", "emotional_tone",
    "objective", "primary_theme", "target_age_segment", "target_os",
    "countries", "hq_region", "kpi_goal",
    
    # === Dates ===
    "creative_launch_date", "campaign_start_date", "campaign_end_date",
    
    # === Visual / content features ===
    "width", "height", "duration_sec",
    "text_density", "copy_length_chars", "readability_score",
    "brand_visibility_score", "clutter_score", "novelty_score", "motion_score",
    "faces_count", "product_count",
    "has_price", "has_discount_badge", "has_gameplay", "has_ugc_style",
    "headline", "subhead", "asset_file",
    
    # === Lifetime raw metrics ===
    "total_days_active", "total_impressions", "total_clicks",
    "total_conversions", "total_spend_usd", "total_revenue_usd",
    "daily_budget_usd",
    
    # === Lifetime derived metrics ===
    "overall_ctr", "overall_cvr", "overall_ipm", "overall_roas",
    "cpc_usd", "cpa_usd",
    
    # === Fatigue metrics ===
    "first_7d_ctr", "last_7d_ctr", "ctr_decay_pct",
    "first_7d_cvr", "last_7d_cvr", "cvr_decay_pct",
    "peak_rolling_ctr_5", "fatigue_day",
    "first_7d_impressions", "first_7d_clicks", "first_7d_conversions",
    "last_7d_impressions", "last_7d_clicks", "last_7d_conversions",
    "peak_day_impressions",
    
    # === Synthetic ground truth (for validation only) ===
    "creative_status", "perf_score",
    
    # === Our derived columns ===
    "ctr_pct_rank_in_vertical", "roas_pct_rank_in_vertical",
    "health_score", "is_fatigued_recoverable", "recommended_action",
]

# Keep only columns that exist; warn about any missing
present  = [c for c in COL_ORDER if c in master.columns]
missing  = [c for c in COL_ORDER if c not in master.columns]
leftover = [c for c in master.columns if c not in COL_ORDER]

if missing:
    print(f"\u26a0  Columns in COL_ORDER but not in master: {missing}")
if leftover:
    print(f"\u2139  Columns in master but not in COL_ORDER (will be appended at end):\n   {leftover}")

master = master[present + leftover]
print(f"\nFinal master shape: {master.shape}")

ℹ  Columns in master but not in COL_ORDER (will be appended at end):
   ['ctr_recomputed', 'cvr_recomputed', 'roas_recomputed', 'ipm']

Final master shape: (1080, 81)


## 6. Write outputs

In [7]:
out_path = OUT_DIR / "master_creatives.csv"
master.to_csv(out_path, index=False)
size_mb = out_path.stat().st_size / 1e6
print(f"\u2713 Wrote {out_path}  ({size_mb:.2f} MB, {len(master):,} rows \u00d7 {master.shape[1]} cols)")

# Smoke test: reload and verify
reloaded = pd.read_csv(out_path)
assert len(reloaded) == EXPECTED_ROWS, "Reload row-count mismatch"
print(f"\u2713 Reload smoke-test passed: {len(reloaded):,} rows")
print("\nFirst 2 rows preview:")
print(reloaded.head(2).T)

✓ Wrote master_creatives.csv  (0.85 MB, 1,080 rows × 81 cols)
✓ Reload smoke-test passed: 1,080 rows

First 2 rows preview:
                                     0                   1
creative_id                     500000              500001
campaign_id                      20000               20000
advertiser_id                     1000                1000
advertiser_name             PixelForge          PixelForge
app_name            PixelForge Legends  PixelForge Legends
...                                ...                 ...
recommended_action                TEST               PAUSE
ctr_recomputed                0.003894            0.006688
cvr_recomputed                 0.19388            0.222317
roas_recomputed               0.673222            1.126695
ipm                           0.754949            1.486797

[81 rows x 2 columns]


In [8]:
# Schema doc for the team
lines = []
lines.append("# master_creatives.csv \u2014 Schema\n")
lines.append(f"_Generated: {datetime.now().isoformat(timespec='seconds')}_\n")
lines.append(f"**{len(master):,} rows \u00d7 {master.shape[1]} columns** (one row per `creative_id`).\n")
lines.append("\n## Column groups\n")

groups = [
    ("IDs",                ["creative_id", "campaign_id", "advertiser_id"]),
    ("Names",              ["advertiser_name", "app_name"]),
    ("Dimensions",         ["vertical", "format", "language", "theme", "hook_type",
                            "cta_text", "dominant_color", "emotional_tone",
                            "objective", "primary_theme", "target_age_segment",
                            "target_os", "countries", "hq_region", "kpi_goal"]),
    ("Dates",              ["creative_launch_date", "campaign_start_date", "campaign_end_date"]),
    ("Visual features",    ["width", "height", "duration_sec", "text_density",
                            "copy_length_chars", "readability_score",
                            "brand_visibility_score", "clutter_score",
                            "novelty_score", "motion_score", "faces_count", "product_count",
                            "has_price", "has_discount_badge", "has_gameplay",
                            "has_ugc_style", "headline", "subhead", "asset_file"]),
    ("Lifetime raw",       ["total_days_active", "total_impressions", "total_clicks",
                            "total_conversions", "total_spend_usd",
                            "total_revenue_usd", "daily_budget_usd"]),
    ("Lifetime derived",   ["overall_ctr", "overall_cvr", "overall_ipm",
                            "overall_roas", "cpc_usd", "cpa_usd"]),
    ("Fatigue metrics",    ["first_7d_ctr", "last_7d_ctr", "ctr_decay_pct",
                            "first_7d_cvr", "last_7d_cvr", "cvr_decay_pct",
                            "peak_rolling_ctr_5", "fatigue_day"]),
    ("Synthetic ground truth (validation only)", ["creative_status", "perf_score"]),
    ("Our derived columns", ["ctr_pct_rank_in_vertical", "roas_pct_rank_in_vertical",
                              "health_score", "is_fatigued_recoverable", "recommended_action"]),
]
for group_name, cols in groups:
    lines.append(f"### {group_name}")
    for c in cols:
        if c in master.columns:
            dtype = str(master[c].dtype)
            n_null = master[c].isnull().sum()
            null_str = f" (nulls: {n_null})" if n_null > 0 else ""
            lines.append(f"- `{c}`  *{dtype}*{null_str}")
    lines.append("")

lines.append("## Key derived columns explained\n")
lines.append("- **`health_score`** (0\u20131): blend of in-vertical CTR rank (40%), ROAS rank (40%), "
             "and decay component (20%). Replaces the opaque synthetic `perf_score` with a transparent score.")
lines.append("- **`is_fatigued_recoverable`** (0/1): creative is flagged `fatigued` AND its lifetime CTR "
             "and ROAS are above the median for stable creatives. These are former winners worth refreshing.")
lines.append("- **`recommended_action`**: SCALE / REFRESH / PAUSE / HOLD / TEST. Rules-based v0; Member 2 "
             "can swap in a model.\n")

lines.append("## Row count contract\n")
lines.append(f"`master_creatives.csv` MUST have exactly {EXPECTED_ROWS:,} rows. "
             "If your downstream code produces a different count after a join, you have a bug.\n")

Path("MASTER_SCHEMA.md").write_text("\n".join(lines))
print("\u2713 Wrote MASTER_SCHEMA.md")

✓ Wrote MASTER_SCHEMA.md


## 7. Quick-look: top performers, recoverables, and action mix

Sanity-check the derived columns by eyeballing a few rows.

In [9]:
print("=== Top 10 by health_score ===\n")
cols_to_show = ["creative_id", "vertical", "format", "creative_status",
                "overall_ctr", "overall_roas", "ctr_decay_pct",
                "health_score", "recommended_action"]
print(master.nlargest(10, "health_score")[cols_to_show].to_string(index=False))

print("\n\n=== Sample of fatigued-but-recoverable creatives (your refresh targets) ===\n")
recov = master[master["is_fatigued_recoverable"] == 1]
print(recov[cols_to_show].head(10).to_string(index=False))

print("\n\n=== Action mix by vertical ===\n")
print(pd.crosstab(master["vertical"], master["recommended_action"]).fillna(0).astype(int))

=== Top 10 by health_score ===

 creative_id      vertical         format creative_status  overall_ctr  overall_roas  ctr_decay_pct  health_score recommended_action
      500898 entertainment         native   top_performer     0.007131      4.214948      -0.581697         0.875              SCALE
      500507        gaming       playable   top_performer     0.013554      2.401432      -0.573680         0.872              SCALE
      500883        travel   interstitial   top_performer     0.009735     11.654016      -0.570168         0.870              SCALE
      500825 entertainment   interstitial          stable     0.007367      3.302327      -0.615270         0.866               HOLD
      500634        travel rewarded_video   top_performer     0.010136     12.311159      -0.631039         0.863              SCALE
      500371     ecommerce   interstitial   top_performer     0.009009      3.218803      -0.594223         0.861              SCALE
      500204 food_delivery         na

## 8. Done-when checklist

- [ ] §2 row count assertions all passed (master = 1,080)
- [ ] §3: ctr/cvr/roas recomputed mismatches are 0 or trivially small (rounding)
- [ ] §4: action mix looks reasonable (no weird 100% concentration)
- [ ] §6: `master_creatives.csv` written and reload smoke-test passed
- [ ] `MASTER_SCHEMA.md` committed and pinned in team chat
- [ ] Tell Member 2 (ML), Member 3 (frontend), Member 4 (insights) in chat: *"master_creatives.csv is ready, schema doc in repo, key columns to know about: `health_score`, `is_fatigued_recoverable`, `recommended_action`"*

**Green → start Phase 3 (`03_daily_fatigue_table.ipynb`).**